# Exploratory Data Analysis 3.0

> Goal: Engineer rolling win-rate features from the processed match dataset and export a feature-ready CSV for model training.

## 1) Setup and Load Processed Data

In [1]:
import pandas as pd

# Load the processed match dataset (one row per game, blue vs red team).
matches_df = pd.read_csv('../data/processed/processed_matches.csv')

## 2) Build Rolling Win-Rate Features

For each match, compute each team's win rate **before** that game is played.  
This avoids data leakage — the model only ever sees historical information at prediction time.

- `blue_team_wr` / `red_team_wr` — rolling win rate (defaults to 0.5 for a team's debut game)
- `blue_team_games` / `red_team_games` — number of games played so far (useful for confidence weighting)

In [2]:
# Sort chronologically so rolling stats accumulate in the correct order.
matches_df = matches_df.sort_values(by="date").reset_index(drop=True)

# Running totals keyed by team name — updated after each match is processed.
team_stats = {}

# Collect enriched rows (original fields + computed win-rate features).
feature_rows = []

for _, row in matches_df.iterrows():
    blue = row["blue_team"]
    red = row["red_team"]

    # First appearance: seed with zero wins/games so the team exists in the dict.
    if blue not in team_stats:
        team_stats[blue] = {"wins": 0, "games": 0}
    if red not in team_stats:
        team_stats[red] = {"wins": 0, "games": 0}

    # Snapshot games played BEFORE this match to avoid leaking current-game result.
    blue_games = team_stats[blue]["games"]
    red_games = team_stats[red]["games"]

    # Default to 0.5 (neutral prior) for a team's debut to avoid division by zero.
    blue_wr = team_stats[blue]["wins"] / blue_games if blue_games > 0 else 0.5
    red_wr = team_stats[red]["wins"] / red_games if red_games > 0 else 0.5

    # Append the original row plus the four new engineered features.
    feature_rows.append({
        **row,
        "blue_team_wr": blue_wr,
        "red_team_wr": red_wr,
        "blue_team_games": blue_games,
        "red_team_games": red_games
    })

    # Update win counts AFTER features have been recorded (keeps stats lag-1).
    if row["blue_side_win"] == 1:
        team_stats[blue]["wins"] += 1
    else:
        team_stats[red]["wins"] += 1

    team_stats[blue]["games"] += 1
    team_stats[red]["games"] += 1

feature_df = pd.DataFrame(feature_rows)

## 3) Inspect Feature Dataset

Validate the new columns — check distributions, spot cold-start rows (debut games), and confirm nothing looks off before export.

In [3]:
# Preview the first few rows to confirm the new columns were added correctly.
feature_df.head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win,blue_team_wr,red_team_wr,blue_team_games,red_team_games
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0,0.5,0.5,0,0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1,0.5,0.5,0,0
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0,0.5,0.5,0,0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1,0.5,0.5,0,0
4,LOLTMNT05_171066,2026-01-09 17:09:20,16.01,LIT,HMBLE,P11 Esports,1,0.0,1.0,1,1


In [4]:
# Summary stats for win-rate columns: confirm range is [0, 1] and mean is near 0.5.
feature_df[["blue_team_wr", "red_team_wr"]].describe()

,blue_team_wr,red_team_wr
count,2792.000000,2792.000000
mean,0.544443,0.527860
std,0.253259,0.248305
min,0.000000,0.000000
25%,0.400000,0.394737
50%,0.538462,0.500000
75%,0.700000,0.666667
max,1.000000,1.000000


In [5]:
# Inspect debut-game rows (games_played == 0) where win rate defaulted to 0.5.
feature_df[feature_df["blue_team_games"] == 0].head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win,blue_team_wr,red_team_wr,blue_team_games,red_team_games
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0,0.5,0.5,0,0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1,0.5,0.5,0,0
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0,0.5,0.5,0,0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1,0.5,0.5,0,0
8,LOLTMNT03_335584,2026-01-12 05:10:10,16.01,LCKC,Nongshim Esports Academy,DN SOOPers Challengers,1,0.5,0.5,0,0


## 4) Export Feature Dataset

Persist the feature-enriched match table so training scripts in `ml/` have a stable, model-ready input file.

In [6]:
# Save the feature-enriched dataset for use in model training and evaluation.
feature_df.to_csv("../data/processed/feature_matches.csv", index=False)

In [7]:
feature_df[["blue_team_wr", "red_team_wr"]].describe()

,blue_team_wr,red_team_wr
count,2792.000000,2792.000000
mean,0.544443,0.527860
std,0.253259,0.248305
min,0.000000,0.000000
25%,0.400000,0.394737
50%,0.538462,0.500000
75%,0.700000,0.666667
max,1.000000,1.000000
